# Turkish Legal Hybrid Retrieval Baseline

## Purpose

This notebook implements a hybrid retrieval baseline that combines dense and lexical (BM25) retrieval.

**Architecture:**
```
Question → Dense Search + BM25 Search → Hybrid Merge → Top-K Results
```

**What this notebook does:**
- Loads the existing FAISS index and embeddings from notebook 03
- Builds a fresh BM25 index on the retrieval corpus
- Implements dense retrieval function using loaded FAISS index
- Implements BM25 retrieval function
- Combines results using normalized score fusion (alpha parameter)
- Tests hybrid retrieval on Turkish legal queries
- Compares dense vs BM25 vs hybrid results
- Saves hybrid retrieval results

**What this notebook does NOT do:**
- Rebuild embeddings or FAISS index (reuses artifacts from notebook 03)
- Implement reranking or cross-encoder
- Generate answers using an LLM
- Evaluate retrieval performance formally

**Hybrid Scoring:**
```
hybrid_score = alpha * dense_score_normalized + (1 - alpha) * bm25_score_normalized
```

Default alpha = 0.5 (equal weight), adjustable for tuning.

## 1. Environment Setup

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

## 2. Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✓ Google Drive mounted")

## 3. Configuration and Paths

In [ ]:
# ===== KAGGLE-ONLY BASELINE =====
# Project configuration
PROJECT_ROOT = "/content/drive/My Drive/nlp-rag-project"
RETRIEVAL_DATA_PATH = f"{PROJECT_ROOT}/data/retrieval/kaggle_retrieval_corpus.csv"
DENSE_OUTPUT_DIR = f"{PROJECT_ROOT}/outputs/dense_retrieval"
HYBRID_OUTPUT_DIR = f"{PROJECT_ROOT}/outputs/hybrid_retrieval"

# Retrieval configuration
TOP_K = 5                    # Final number of results to return
DENSE_CANDIDATES = 20        # Candidates to retrieve from dense search
BM25_CANDIDATES = 20         # Candidates to retrieve from BM25 search
ALPHA = 0.5                  # Hybrid weight: alpha*dense + (1-alpha)*bm25

# Create output directory
Path(HYBRID_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Retrieval corpus: {RETRIEVAL_DATA_PATH}")
print(f"Dense artifacts: {DENSE_OUTPUT_DIR}")
print(f"Output directory: {HYBRID_OUTPUT_DIR}")
print(f"\nHybrid configuration:")
print(f"  - Top K: {TOP_K}")
print(f"  - Dense candidates: {DENSE_CANDIDATES}")
print(f"  - BM25 candidates: {BM25_CANDIDATES}")
print(f"  - Alpha (dense weight): {ALPHA}")

## 4. Install and Import Dependencies

In [ ]:
import subprocess
import sys

print("Installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rank-bm25"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
print("✓ Dependencies installed")

In [ ]:
from rank_bm25 import BM25Okapi
import faiss

print("✓ All imports successful")

## 5. Load Retrieval Corpus

In [ ]:
print(f"Loading retrieval corpus...")
df_corpus = pd.read_csv(RETRIEVAL_DATA_PATH)

print(f"✓ Loaded retrieval corpus")
print(f"  - Shape: {df_corpus.shape}")
print(f"  - Columns: {list(df_corpus.columns)[:5]}...")
print(f"  - Non-null chunk_text: {df_corpus['chunk_text'].notna().sum()}")

## 6. Load Dense Retrieval Artifacts

In [ ]:
# Load FAISS index
index_path = f"{DENSE_OUTPUT_DIR}/kaggle_faiss_index.index"
print(f"Loading FAISS index from {index_path}...")
faiss_index = faiss.read_index(index_path)
print(f"✓ FAISS index loaded")
print(f"  - Index type: {type(faiss_index).__name__}")
print(f"  - Number of vectors: {faiss_index.ntotal}")
print(f"  - Vector dimension: {faiss_index.d}")

In [ ]:
# Load embeddings for query encoding
embeddings_path = f"{DENSE_OUTPUT_DIR}/kaggle_dense_embeddings.npy"
print(f"Loading embeddings from {embeddings_path}...")
embeddings = np.load(embeddings_path)
print(f"✓ Embeddings loaded")
print(f"  - Shape: {embeddings.shape}")
print(f"  - Data type: {embeddings.dtype}")

In [ ]:
# Load dense metadata for embedding model info
metadata_path = f"{DENSE_OUTPUT_DIR}/kaggle_dense_retrieval_metadata.json"
with open(metadata_path, 'r', encoding='utf-8') as f:
    dense_metadata = json.load(f)

print(f"Dense retrieval metadata:")
print(f"  - Model: {dense_metadata['embedding_model']}")
print(f"  - Dimension: {dense_metadata['embedding_dimension']}")
print(f"  - Corpus size: {dense_metadata['corpus_size']}")

In [ ]:
# Load row mapping for FAISS → DataFrame indexing
mapping_path = f"{DENSE_OUTPUT_DIR}/kaggle_retrieval_row_mapping.csv"
df_mapping = pd.read_csv(mapping_path)
print(f"✓ Row mapping loaded")
print(f"  - Shape: {df_mapping.shape}")

# Create reverse mapping (corpus index → FAISS index)
faiss_index_map = dict(zip(df_mapping['corpus_row_index'], df_mapping['faiss_index']))
print(f"  - Mapping entries: {len(faiss_index_map)}")

## 7. Build BM25 Index

In [ ]:
def preprocess_text(text: str) -> List[str]:
    """
    Preprocess text for BM25:
    - lowercase
    - strip whitespace
    - collapse repeated spaces
    - tokenize on whitespace
    - preserve Turkish characters
    """
    if not isinstance(text, str):
        return []
    
    # Lowercase
    text = text.lower()
    
    # Strip and collapse whitespace
    text = " ".join(text.split())
    
    # Tokenize on whitespace
    tokens = text.split()
    
    return tokens


print("Building BM25 index...")

# Preprocess all corpus texts
corpus_tokens = []
for text in df_corpus['chunk_text']:
    tokens = preprocess_text(text)
    corpus_tokens.append(tokens)

# Build BM25 index
bm25_index = BM25Okapi(corpus_tokens)

print(f"✓ BM25 index built")
print(f"  - Corpus size: {len(corpus_tokens)}")
print(f"  - Vocabulary size: {len(bm25_index.idf)}")
print(f"  - Avg document length: {np.mean([len(doc) for doc in corpus_tokens]):.1f} tokens")

## 8. Implement Dense Retrieval Helper

In [ ]:
def retrieve_dense_candidates(
    query: str,
    model,  # SentenceTransformer model from notebook 03
    index: faiss.Index,
    corpus_df: pd.DataFrame,
    top_k: int = DENSE_CANDIDATES
) -> Dict[str, Tuple[float, int]]:
    """
    Retrieve candidate chunks using dense retrieval.
    Returns: dict mapping chunk_id → (similarity_score, corpus_row_index)
    """
    # Need to load model for encoding
    from sentence_transformers import SentenceTransformer
    if model is None:
        embedding_model = dense_metadata['embedding_model']
        model = SentenceTransformer(embedding_model)
    
    # Encode query
    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )
    query_embedding = np.array([query_embedding], dtype=np.float32)
    
    # Search FAISS index
    scores, faiss_indices = index.search(query_embedding, top_k)
    
    # Map FAISS indices back to corpus indices and collect results
    results = {}
    for faiss_idx, score in zip(faiss_indices[0], scores[0]):
        corpus_idx = df_mapping.iloc[faiss_idx]['corpus_row_index']
        chunk_id = df_mapping.iloc[faiss_idx]['chunk_id']
        results[chunk_id] = (float(score), int(corpus_idx))
    
    return results


print("✓ Dense retrieval function defined")

## 9. Implement BM25 Retrieval Helper

In [ ]:
def retrieve_bm25_candidates(
    query: str,
    bm25: BM25Okapi,
    corpus_df: pd.DataFrame,
    top_k: int = BM25_CANDIDATES
) -> Dict[str, Tuple[float, int]]:
    """
    Retrieve candidate chunks using BM25 lexical retrieval.
    Returns: dict mapping chunk_id → (bm25_score, corpus_row_index)
    """
    # Preprocess query
    query_tokens = preprocess_text(query)
    
    # Get BM25 scores for all documents
    bm25_scores = bm25.get_scores(query_tokens)
    
    # Get top-k indices
    top_indices = np.argsort(bm25_scores)[::-1][:top_k]
    
    # Map back to corpus and collect results
    results = {}
    for corpus_idx in top_indices:
        score = float(bm25_scores[corpus_idx])
        if corpus_idx < len(corpus_df):
            chunk_id = corpus_df.iloc[corpus_idx]['chunk_id']
            results[chunk_id] = (score, corpus_idx)
    
    return results


print("✓ BM25 retrieval function defined")

## 10. Implement Hybrid Merge and Scoring

In [ ]:
def retrieve_hybrid(
    query: str,
    model,
    faiss_index: faiss.Index,
    bm25: BM25Okapi,
    corpus_df: pd.DataFrame,
    top_k: int = TOP_K,
    alpha: float = ALPHA,
    dense_candidates: int = DENSE_CANDIDATES,
    bm25_candidates: int = BM25_CANDIDATES
) -> pd.DataFrame:
    """
    Perform hybrid retrieval by combining dense and BM25 results.
    
    Args:
        alpha: weight for dense score (1-alpha for BM25)
    
    Returns:
        DataFrame with top-k hybrid results
    """
    # Get dense candidates
    dense_results = retrieve_dense_candidates(
        query, model, faiss_index, corpus_df, dense_candidates
    )
    
    # Get BM25 candidates
    bm25_results = retrieve_bm25_candidates(
        query, bm25, corpus_df, bm25_candidates
    )
    
    # Collect all unique chunks
    all_chunks = set(dense_results.keys()) | set(bm25_results.keys())
    
    # Normalize scores to [0, 1]
    dense_scores = [score for score, _ in dense_results.values()]
    bm25_scores = [score for score, _ in bm25_results.values()]
    
    dense_min, dense_max = min(dense_scores) if dense_scores else 0, max(dense_scores) if dense_scores else 1
    bm25_min, bm25_max = min(bm25_scores) if bm25_scores else 0, max(bm25_scores) if bm25_scores else 1
    
    dense_range = dense_max - dense_min if dense_max > dense_min else 1
    bm25_range = bm25_max - bm25_min if bm25_max > bm25_min else 1
    
    # Compute hybrid scores
    hybrid_scores = {}
    for chunk_id in all_chunks:
        # Get dense score (normalized)
        if chunk_id in dense_results:
            dense_score, corpus_idx = dense_results[chunk_id]
            dense_norm = (dense_score - dense_min) / dense_range if dense_range > 0 else 0
        else:
            dense_norm = 0
            corpus_idx = bm25_results[chunk_id][1]
        
        # Get BM25 score (normalized)
        if chunk_id in bm25_results:
            bm25_score, _ = bm25_results[chunk_id]
            bm25_norm = (bm25_score - bm25_min) / bm25_range if bm25_range > 0 else 0
        else:
            bm25_norm = 0
        
        # Hybrid score
        hybrid_score = alpha * dense_norm + (1 - alpha) * bm25_norm
        
        hybrid_scores[chunk_id] = {
            'hybrid_score': hybrid_score,
            'dense_score': dense_results.get(chunk_id, (0, corpus_idx))[0],
            'bm25_score': bm25_results.get(chunk_id, (0, corpus_idx))[0],
            'corpus_idx': corpus_idx
        }
    
    # Sort by hybrid score
    sorted_chunks = sorted(
        hybrid_scores.items(),
        key=lambda x: x[1]['hybrid_score'],
        reverse=True
    )[:top_k]
    
    # Build result dataframe
    results = []
    for rank, (chunk_id, scores) in enumerate(sorted_chunks, 1):
        corpus_idx = scores['corpus_idx']
        row = corpus_df.iloc[corpus_idx].to_dict()
        row['rank'] = rank
        row['hybrid_score'] = scores['hybrid_score']
        row['dense_score'] = scores['dense_score']
        row['bm25_score'] = scores['bm25_score']
        results.append(row)
    
    return pd.DataFrame(results)


print("✓ Hybrid retrieval function defined")

## 11. Load Embedding Model for Query Encoding

In [ ]:
print(f"Loading embedding model for query encoding...")
from sentence_transformers import SentenceTransformer

embedding_model_name = dense_metadata['embedding_model']
embedding_model = SentenceTransformer(embedding_model_name)

print(f"✓ Embedding model loaded: {embedding_model_name}")

## 12. Test Queries

In [ ]:
# Define same Turkish legal test queries as before
test_queries = [
    "Haksız zenginleşme ile ilgili hükümler nelerdir?",
    "Miras bırakanın tasarruf özgürlüğü nasıl sınırlandırılır?",
    "Ceza muhakemesinde tutuklama şartları nelerdir?",
    "Anayasa'ya göre devletin şekli nedir?",
    "Aile yurdu ile ilgili malik üzerindeki sınırlamalar nelerdir?"
]

print(f"Defined {len(test_queries)} test queries")

In [ ]:
# Run hybrid retrieval for each test query
all_hybrid_results = []
overlap_stats = []

print("="*80)
print("RUNNING HYBRID RETRIEVAL TEST QUERIES")
print("="*80)

for query_idx, query in enumerate(test_queries, 1):
    print(f"\n{'='*80}")
    print(f"Query {query_idx}: {query}")
    print(f"{'='*80}")
    
    # Retrieve hybrid results
    hybrid_results = retrieve_hybrid(
        query=query,
        model=embedding_model,
        faiss_index=faiss_index,
        bm25=bm25_index,
        corpus_df=df_corpus,
        top_k=TOP_K,
        alpha=ALPHA,
        dense_candidates=DENSE_CANDIDATES,
        bm25_candidates=BM25_CANDIDATES
    )
    
    # Add query info
    hybrid_results['test_query_index'] = query_idx
    hybrid_results['test_query_text'] = query
    all_hybrid_results.append(hybrid_results)
    
    # Get dense and BM25 only results for comparison
    dense_only = retrieve_dense_candidates(query, embedding_model, faiss_index, df_corpus, 3)
    bm25_only = retrieve_bm25_candidates(query, bm25_index, df_corpus, 3)
    
    # Print hybrid results
    print(f"\nTop {TOP_K} Hybrid Results:")
    for _, row in hybrid_results.iterrows():
        print(f"\n  [{row['rank']}] Hybrid: {row['hybrid_score']:.4f} | Dense: {row['dense_score']:.4f} | BM25: {row['bm25_score']:.4f}")
        print(f"      Source: {row['source']}")
        print(f"      Chunk: {row['chunk_text'][:120]}...")
    
    # Show comparison
    print(f"\nComparison - Dense Only (top 3):")
    for idx, (chunk_id, (score, cidx)) in enumerate(list(dense_only.items())[:3], 1):
        chunk_text = df_corpus.iloc[cidx]['chunk_text']
        print(f"  {idx}. Score: {score:.4f} | {chunk_text[:100]}...")
    
    print(f"\nComparison - BM25 Only (top 3):")
    for idx, (chunk_id, (score, cidx)) in enumerate(list(bm25_only.items())[:3], 1):
        chunk_text = df_corpus.iloc[cidx]['chunk_text']
        print(f"  {idx}. Score: {score:.4f} | {chunk_text[:100]}...")
    
    # Calculate overlap
    dense_chunks = set(dense_only.keys())
    bm25_chunks = set(bm25_only.keys())
    overlap = len(dense_chunks & bm25_chunks)
    overlap_stats.append((query_idx, overlap, len(dense_chunks) + len(bm25_chunks) - overlap))
    print(f"\nOverlap: {overlap}/3 chunks in both dense and BM25 top-3")

In [ ]:
# Combine all results
df_hybrid_results = pd.concat(all_hybrid_results, ignore_index=True)

print(f"\n✓ Completed all hybrid retrieval queries")
print(f"  - Total results: {len(df_hybrid_results)}")
print(f"  - Results per query: {len(df_hybrid_results) // len(test_queries)}")

print(f"\nOverlap Statistics (dense vs BM25 in top-3 candidates):")
for query_idx, overlap, union in overlap_stats:
    print(f"  Query {query_idx}: {overlap}/3 overlap, {union} unique candidates")

## 13. Save Hybrid Results

In [ ]:
# Reorder columns for clarity
column_order = [
    'rank', 'hybrid_score', 'dense_score', 'bm25_score',
    'chunk_id', 'doc_id',
    'source', 'category',
    'question', 'answer',
    'chunk_text',
    'test_query_index', 'test_query_text'
]
return_cols = [c for c in column_order if c in df_hybrid_results.columns]
df_hybrid_results = df_hybrid_results[return_cols]

print(f"✓ Finalized hybrid results DataFrame")
print(f"  Shape: {df_hybrid_results.shape}")

In [ ]:
# Save as CSV
csv_output = f"{HYBRID_OUTPUT_DIR}/kaggle_hybrid_retrieval_results.csv"
df_hybrid_results.to_csv(csv_output, index=False)
print(f"✓ Saved: {csv_output}")
print(f"  Size: {Path(csv_output).stat().st_size / 1024:.2f} KB")

In [ ]:
# Save as JSONL
jsonl_output = f"{HYBRID_OUTPUT_DIR}/kaggle_hybrid_retrieval_results.jsonl"
with open(jsonl_output, 'w', encoding='utf-8') as f:
    for idx, row in df_hybrid_results.iterrows():
        json_record = row.to_dict()
        json_record = {
            k: (None if pd.isna(v) else v) for k, v in json_record.items()
        }
        f.write(json.dumps(json_record, ensure_ascii=False) + "\n")

print(f"✓ Saved: {jsonl_output}")
print(f"  Size: {Path(jsonl_output).stat().st_size / 1024:.2f} KB")

In [ ]:
# Save metadata
metadata = {
    'corpus_size': len(df_corpus),
    'top_k': TOP_K,
    'dense_candidates': DENSE_CANDIDATES,
    'bm25_candidates': BM25_CANDIDATES,
    'alpha': ALPHA,
    'embedding_model': embedding_model_name,
    'bm25_corpus_size': len(corpus_tokens),
    'faiss_index_size': faiss_index.ntotal,
    'test_queries_count': len(test_queries),
    'total_results': len(df_hybrid_results),
    'files': {
        'kaggle_hybrid_retrieval_results_csv': 'kaggle_hybrid_retrieval_results.csv',
        'kaggle_hybrid_retrieval_results_jsonl': 'kaggle_hybrid_retrieval_results.jsonl',
        'kaggle_hybrid_retrieval_metadata': 'kaggle_hybrid_retrieval_metadata.json'
    }
}

metadata_path = f"{HYBRID_OUTPUT_DIR}/kaggle_hybrid_retrieval_metadata.json"
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"✓ Saved: {metadata_path}")

In [ ]:
# Optional: Save comparison data
comparison_data = []
for idx, row in df_hybrid_results.iterrows():
    comparison_data.append({
        'test_query_index': row['test_query_index'],
        'rank': row['rank'],
        'chunk_id': row['chunk_id'],
        'hybrid_score': row['hybrid_score'],
        'dense_score': row['dense_score'],
        'bm25_score': row['bm25_score'],
        'dense_rank': df_hybrid_results[(df_hybrid_results['test_query_index'] == row['test_query_index']) & 
                                          (df_hybrid_results['dense_score'] > 0)]['dense_score'].rank(ascending=False).get(idx, np.nan),
        'bm25_rank': df_hybrid_results[(df_hybrid_results['test_query_index'] == row['test_query_index']) & 
                                         (df_hybrid_results['bm25_score'] > 0)]['bm25_score'].rank(ascending=False).get(idx, np.nan)
    })

df_comparison = pd.DataFrame(comparison_data)
comparison_path = f"{HYBRID_OUTPUT_DIR}/kaggle_dense_vs_bm25_vs_hybrid_comparison.csv"
df_comparison.to_csv(comparison_path, index=False)
print(f"✓ Saved comparison: {comparison_path}")

In [ ]:
print("\n" + "="*80)
print("HYBRID RETRIEVAL BASELINE COMPLETE")
print("="*80)

print(f"\nRetrieval System Statistics:")
print(f"  • Corpus size: {len(df_corpus):,} chunks")
print(f"  • Dense encoding model: {embedding_model_name}")
print(f"  • Dense index size (FAISS): {faiss_index.ntotal}")
print(f"  • BM25 corpus size: {len(corpus_tokens)}")
print(f"  • Vocabulary size: {len(bm25_index.idf)}")

print(f"\nHybrid Configuration:")
print(f"  • Top K: {TOP_K}")
print(f"  • Dense candidates: {DENSE_CANDIDATES}")
print(f"  • BM25 candidates: {BM25_CANDIDATES}")
print(f"  • Alpha (dense weight): {ALPHA}")
print(f"  • Hybrid weight (BM25): {1 - ALPHA}")

print(f"\nTest Results:")
print(f"  • Test queries: {len(test_queries)}")
print(f"  • Total retrieval results: {len(df_hybrid_results)}")
print(f"  • Results per query: {len(df_hybrid_results) // len(test_queries)}")

print(f"\nOutput Files (saved to {HYBRID_OUTPUT_DIR}):")
output_files = [
    ("kaggle_hybrid_retrieval_results.csv", "Kaggle-only hybrid retrieval results for 5 test queries"),
    ("kaggle_hybrid_retrieval_results.jsonl", "Same results in JSONL format"),
    ("kaggle_hybrid_retrieval_metadata.json", "Metadata about Kaggle-only hybrid configuration"),
    ("kaggle_dense_vs_bm25_vs_hybrid_comparison.csv", "Side-by-side comparison of Kaggle-only retrieval methods"),
]

for fname, desc in output_files:
    path = f"{HYBRID_OUTPUT_DIR}/{fname}"
    if Path(path).exists():
        size = Path(path).stat().st_size
        size_str = f"{size / (1024*1024):.2f} MB" if size > 1024*1024 else f"{size / 1024:.2f} KB"
        print(f"  ✓ {fname}")
        print(f"    {desc} ({size_str})")

print(f"\nArchitecture Progress:")
print(f"  ✓ Step 1: Dataset preparation")
print(f"  ✓ Step 2: Retrieval corpus preparation")
print(f"  ✓ Step 3: Dense retrieval baseline")
print(f"  ✓ Step 4: Hybrid retrieval baseline (this notebook)")
print(f"  → Step 5: Optional reranking layer (cross-encoder)")
print(f"  → Step 6: LLM answer generation")
print(f"  → Step 7: End-to-end RAG evaluation benchmark")

print(f"\nNext Steps:")
print(f"  1. Analyze hybrid vs dense vs BM25 results (comparison CSV)")
print(f"  2. Tune alpha parameter (currently {ALPHA}) for better results")
print(f"  3. Implement optional reranking (cross-encoder) over hybrid results")
print(f"  4. Integrate LLM for answer generation")
print(f"  5. Create formal benchmark for end-to-end RAG evaluation")
print(f"  6. Deploy as API or Gradio interface")

print(f"\nNotes:")
print(f"  • Hybrid scoring uses normalized score fusion (min-max normalization)")
print(f"  • Dense and BM25 scores are independently normalized then fused")
print(f"  • Alpha parameter controls dense-BM25 weight balance")
print(f"  • This is the hybrid retrieval stage only")
print(f"  • No reranking, answer generation, or evaluation is performed yet")